<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/premi-secundaria/++theme++genweb6.theme/img/logo.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Automatización y robotización de procesos</p>
  </div>

</div>

# 04 · Automatización masiva y ejecución programada

## Laboratorio: documentos, códigos de salida y contrato del programador

Caso con datos ficticios. La práctica crea su propio proyecto temporal, produce documentos Markdown y revisa resultados por elemento. **No registra tareas ni carga calendarios en Windows, macOS o Linux.**

Las configuraciones se estudian como archivos. Su ensayo nativo se realiza después, deliberadamente, en un equipo preparado.

**Alumnado:** guía completa y doce tareas por resolver.

## Objetivos

Separar selección, validación y generación; comprender simulación y repetición; interpretar códigos; impedir dos escritores cooperantes; entregar ejecutable, argumentos, directorio y registros al programador del sistema.

Windows recibe el recorrido más detallado de la teoría. El notebook contrasta las partes que pueden ejecutarse de forma portable, sin suponer que PowerShell o launchd estén disponibles.

## Requisitos y estructura

Python 3.11+, biblioteca estándar. Abre Jupyter desde la carpeta extraída del bloque y conserva `procesar_lote.py`, `programacion/` y `recursos/`. La guía define sus funciones para leerlas y explicarlas. El ensayo de CLI llama además al archivo real `procesar_lote.py` incluido.

Los scripts de alumnado y solución deben conservar ese archivo a su lado. No necesitan paquetes adicionales ni instalar planificadores.

## Cómo trabajar

Ejecuta en orden desde un kernel limpio. Completa los doce ejercicios y activa sus banderas. Cada tarea que escribe crea una demo propia con nombre independiente. Repetir una celda de creación puede producir una colisión esperada; no se vacía un directorio anterior.

Las rutas impresas por el solucionario pertenecen a una ejecución de verificación y se generarán de nuevo. No las reutilices como configuración de tu equipo.

In [ ]:
"""Lote docente de documentos sintéticos, Python 3.11+ y biblioteca estándar.

No instala tareas. Cada configuración apunta a su propio proyecto privado.
Códigos: 0 correcto; 10 incidencias; 20 configuración; 30 fallo operativo;
40 bloqueo ocupado. argparse utiliza 2 para argumentos incompatibles.
"""
from pathlib import Path, PureWindowsPath
from contextlib import contextmanager
from datetime import datetime, timezone
from decimal import Decimal
from string import Template
from tempfile import TemporaryDirectory
from collections import Counter
import argparse
import json
import logging
import os
import re
import sys
import uuid
import time

VERSION = "lotes-aula-1.0"
MAX_BYTES = 1024 * 1024
LOG = logging.getLogger("lote")



import subprocess
import plistlib
import configparser

PROYECTO = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
MOTOR = PROYECTO / "procesar_lote.py"
if not MOTOR.is_file():
    raise FileNotFoundError("Abre Jupyter desde la carpeta del bloque con procesar_lote.py")


## 1. Frontera del lote

Se seleccionan JSON regulares, directos y no enlaces. El TXT de explicación se ignora. Un dato inválido recibe un estado; un fallo operativo no se disfraza como un error de contenido.

Trabajamos con tres documentos válidos y dos inválidos, sin modificar las entradas durante el procesamiento.

In [ ]:
class ConfigError(ValueError):
    pass

class BloqueoOcupado(RuntimeError):
    pass


### Crear el proyecto de aula

La raíz debe ser nueva. Se crean carpetas separadas y una plantilla. No se reutilizan ni se borran carpetas existentes. Los datos no corresponden a personas o empresas reales.

In [ ]:
def crear_demo(destino):
    raiz = Path(destino).absolute()
    raiz.mkdir(parents=True, exist_ok=False)
    for nombre in ("entrada", "salida", "informes", "estado", "plantillas"):
        (raiz / nombre).mkdir()
    config = {"version": 1, "entrada": "entrada", "salida": "salida",
              "informes": "informes", "estado": "estado", "plantilla": "plantillas/documento.txt"}
    (raiz / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
    plantilla = "# Informe de aula ${id}\n\nAño: ${anio}\nImporte: ${importe} EUR\nTerritorio ficticio: ${territorio}\nMotor: ${version}\n\nDatos sintéticos. Sin conclusiones tributarias reales.\n"
    (raiz / config["plantilla"]).write_text(plantilla, encoding="utf-8", newline="\n")
    for numero, identificador, importe in [(1, "DEMO-001", "1200000.00"),
                                           (2, "DEMO-002", "800000.00"),
                                           (5, "DEMO-005", "650000.00")]:
        dato = {"id": identificador, "anio": 2025, "importe": importe,
                "territorio": "Territorio DEMO"}
        (raiz / "entrada" / (f"{numero:02d}_valido.json")).write_text(
            json.dumps(dato, ensure_ascii=False), encoding="utf-8")
    (raiz / "entrada/03_json_invalido.json").write_text('{"id":', encoding="utf-8")
    (raiz / "entrada/04_campo_falta.json").write_text('{"id":"DEMO-004"}', encoding="utf-8")
    (raiz / "entrada/LEEME.txt").write_text("Este archivo no forma parte del lote JSON.\n", encoding="utf-8")
    return raiz / "config.json"


In [ ]:
temporal = TemporaryDirectory(prefix="curso_lotes_")
base = Path(temporal.name)
config_demo = crear_demo(base / "proyecto de aula")
entrada_demo = config_demo.parent / "entrada"
entrada_antes = {p.name:p.read_bytes() for p in entrada_demo.iterdir() if p.is_file()}
print("Configuración:", config_demo)
print("Archivos iniciales:", sorted(entrada_antes))


## 2. JSON y tamaño con contrato explícito

El lector admite hasta 1 MiB y UTF-8. El JSON estricto rechaza claves repetidas y constantes como NaN. Una entrada interpretable no es todavía un documento válido: después se comprueban sus campos.

El límite se aplica sobre los bytes leídos, no solo sobre un tamaño consultado antes.

In [ ]:
def json_estricto(texto):
    def pares_sin_duplicar(pares):
        resultado = {}
        for clave, valor in pares:
            if clave in resultado:
                raise ValueError("Clave JSON repetida")
            resultado[clave] = valor
        return resultado
    def constante_invalida(valor):
        raise ValueError("Constante JSON no admitida: " + valor)
    return json.loads(texto, object_pairs_hook=pares_sin_duplicar,
                      parse_constant=constante_invalida)

def leer_acotado(ruta):
    with Path(ruta).open("rb") as archivo:
        contenido = archivo.read(MAX_BYTES + 1)
    if len(contenido) > MAX_BYTES:
        raise ValueError("Archivo superior al límite de aula de 1 MiB")
    return contenido.decode("utf-8")


In [ ]:
dato_inicial = json_estricto(leer_acotado(entrada_demo / "01_valido.json"))
print(dato_inicial)
try:
    json_estricto('{"id":"uno","id":"dos"}')
except ValueError as error:
    print("Rechazo esperado:", error)


## 3. Configuración independiente de cwd

Los caminos internos se interpretan desde la carpeta de config.json. No son rutas libres hacia carpetas de terceros: la práctica rechaza ascensos, enlaces y solapamientos entre entrada, salida, informes y estado.

Este control supone un proyecto privado estable. No es un sandbox contra un proceso hostil que cambie el árbol mientras trabajamos.

In [ ]:
def ruta_interna(raiz, relativa):
    if not isinstance(relativa, str) or not relativa:
        raise ConfigError("Cada ruta configurada debe ser texto relativo")
    p = Path(relativa)
    if p.is_absolute() or PureWindowsPath(relativa).anchor or ".." in p.parts:
        raise ConfigError("No se admiten rutas absolutas ni ascensos")
    actual = raiz
    for parte in p.parts:
        actual = actual / parte
        if actual.is_symlink():
            raise ConfigError("No se admiten enlaces en rutas configuradas")
    resuelta = actual.resolve()
    if not resuelta.is_relative_to(raiz) or resuelta == raiz:
        raise ConfigError("La ruta debe quedar dentro del proyecto")
    return resuelta

def cargar_config(ruta_config):
    original = Path(ruta_config).absolute()
    if original.is_symlink():
        raise ConfigError("La configuración no puede ser un enlace")
    try:
        ruta = original.resolve(strict=True)
    except (FileNotFoundError, NotADirectoryError) as error:
        raise ConfigError("No existe la configuración indicada") from error
    try:
        datos = json_estricto(leer_acotado(ruta))
    except (ValueError, UnicodeError) as error:
        raise ConfigError("Configuración JSON incompatible") from error
    claves = {"version", "entrada", "salida", "informes", "estado", "plantilla"}
    if not isinstance(datos, dict) or set(datos) != claves or type(datos["version"]) is not int or datos["version"] != 1:
        raise ConfigError("Se requiere esquema de configuración versión 1")
    raiz = ruta.parent
    cfg = {k: ruta_interna(raiz, datos[k]) for k in claves - {"version"}}
    for k in ("entrada", "salida", "informes", "estado"):
        if not cfg[k].is_dir():
            raise ConfigError("Falta la carpeta configurada: " + k)
    carpetas = [cfg[k] for k in ("entrada", "salida", "informes", "estado")]
    for i, a in enumerate(carpetas):
        for b in carpetas[i + 1:]:
            if a.is_relative_to(b) or b.is_relative_to(a):
                raise ConfigError("Las carpetas de trabajo no deben solaparse")
    if not cfg["plantilla"].is_file():
        raise ConfigError("Falta el archivo de plantilla")
    cfg.update(raiz=raiz, config=ruta)
    return cfg


In [ ]:
cfg_demo = cargar_config(config_demo)
for k in ("entrada","salida","informes","estado","plantilla"):
    print(k, cfg_demo[k])
assert cfg_demo["entrada"].is_absolute()


## 4. Plantilla y documento validado

La plantilla debe usar exactamente id, anio, importe, territorio y version. Template.substitute requiere esos datos. No insertamos hora de ejecución en el documento para no cambiar su contenido en cada repetición.

El año permitido y el formato del ID son reglas sintéticas del aula, no requisitos legales de publicación tributaria.

In [ ]:
def cargar_plantilla(ruta):
    try:
        plantilla = Template(leer_acotado(ruta))
    except (ValueError, UnicodeError) as error:
        raise ConfigError("No se puede interpretar la plantilla") from error
    permitidos = {"id", "anio", "importe", "territorio", "version"}
    if not plantilla.is_valid() or set(plantilla.get_identifiers()) != permitidos:
        raise ConfigError("La plantilla debe usar exactamente los cinco campos acordados")
    return plantilla

def validar_documento(datos):
    requeridos = {"id", "anio", "importe", "territorio"}
    if not isinstance(datos, dict) or set(datos) != requeridos:
        raise ValueError("Campos incompatibles con el contrato del documento")
    if not isinstance(datos["id"], str) or re.fullmatch(r"DEMO-[0-9]{3}", datos["id"]) is None:
        raise ValueError("ID sintético no admitido")
    if type(datos["anio"]) is not int or datos["anio"] not in {2024, 2025}:
        raise ValueError("Año fuera del contrato de aula")
    importe = datos["importe"]
    if not isinstance(importe, str) or re.fullmatch(r"[0-9]{1,12}(?:\.[0-9]{1,2})?", importe) is None:
        raise ValueError("Formato de importe incompatible")
    importe = Decimal(importe)
    if not importe.is_finite() or importe <= 0:
        raise ValueError("Importe no positivo o no finito")
    territorio = datos["territorio"]
    if not isinstance(territorio, str) or not territorio.strip() or len(territorio) > 80 or any(ord(c) < 32 for c in territorio):
        raise ValueError("Territorio ficticio incompatible")
    return {**datos, "importe": format(importe.quantize(Decimal("0.01")), "f"),
            "territorio": territorio.strip(), "version": VERSION}


In [ ]:
plantilla_demo = cargar_plantilla(cfg_demo["plantilla"])
dato_validado = validar_documento(dato_inicial)
texto_documento = plantilla_demo.substitute(dato_validado)
print(texto_documento)
assert dato_validado["importe"] == "1200000.00"


## 5. Selección y decisión por elemento

Se ordenan nombres para obtener un recorrido comprensible. El nombre de salida usa ID y año validados. Una clave repetida en el lote es duplicada. Una salida existente idéntica se omite; si difiere, se declara conflicto. No se reemplaza automáticamente.

Una escritura exclusiva puede fallar tras crear parte del archivo. Una repetición posterior puede detectar ese resultado como conflicto y exigir revisión.

In [ ]:
def seleccionar(cfg):
    # Solo hijos JSON regulares. No se procesa recursivamente la salida.
    return sorted((p for p in cfg["entrada"].iterdir()
                   if not p.is_symlink() and p.is_file() and p.suffix.lower() == ".json"),
                  key=lambda p: p.name)

def evaluar_elemento(ruta, cfg, plantilla, vistos, simular):
    registro = {"entrada": ruta.name, "salida": None}
    try:
        datos = validar_documento(json_estricto(leer_acotado(ruta)))
    except (ValueError, UnicodeError) as error:
        return {**registro, "estado": "invalido", "motivo": str(error)}
    # Un fallo operativo de lectura/escritura se propaga al nivel de ejecución.
    nombre = "doc_" + datos["id"] + "_" + str(datos["anio"]) + ".md"
    registro["salida"] = nombre
    if nombre in vistos:
        return {**registro, "estado": "duplicado", "motivo": "Clave repetida en el lote"}
    vistos.add(nombre)
    destino = cfg["salida"] / nombre
    if destino.is_symlink():
        raise OSError("La salida no puede ser un enlace")
    contenido = plantilla.substitute(datos).encode("utf-8")
    if destino.exists():
        if not destino.is_file():
            raise OSError("Un destino de documento no es un archivo")
        if destino.stat().st_size == len(contenido) and destino.read_bytes() == contenido:
            return {**registro, "estado": "omitido", "motivo": "Mismo contenido de salida"}
        return {**registro, "estado": "conflicto", "motivo": "El destino tiene contenido distinto"}
    if simular:
        return {**registro, "estado": "previsto", "motivo": "Se crearía un documento nuevo"}
    try:
        with destino.open("xb") as f:
            f.write(contenido)
    except FileExistsError:
        return {**registro, "estado": "conflicto", "motivo": "El destino apareció durante la operación"}
    return {**registro, "estado": "creado", "motivo": "Creación exclusiva terminada"}


In [ ]:
seleccion_demo = seleccionar(cfg_demo)
print([p.name for p in seleccion_demo])
assert len(seleccion_demo) == 5
assert all(p.suffix.lower()==".json" for p in seleccion_demo)


## 6. Bloqueo local cooperante

La creación exclusiva de ejecucion.lock evita dos escritores que comparten el mismo estado. La ejecución guarda identidad, PID y hora. Al finalizar retira únicamente el bloqueo con su token.

Si el proceso muere de forma forzada puede quedar un bloqueo huérfano. No se borra automáticamente según edad o PID. No es un bloqueo distribuido de red.

In [ ]:
@contextmanager
def bloqueo_local(carpeta):
    """Archivo exclusivo, solo para procesos cooperantes de un proyecto local.

    No se recuperan bloqueos huérfanos automáticamente. No es un bloqueo
    distribuido ni una defensa ante modificaciones hostiles del directorio.
    """
    ruta = Path(carpeta) / "ejecucion.lock"
    token = uuid.uuid4().hex
    try:
        fd = os.open(ruta, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)
    except FileExistsError as error:
        raise BloqueoOcupado("Ya existe un bloqueo; revisar la otra ejecución") from error
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            json.dump({"token": token, "pid": os.getpid(),
                       "inicio_utc": datetime.now(timezone.utc).isoformat()}, f)
        yield
    finally:
        # No se elimina un bloqueo que ya no tenga nuestra identidad.
        if ruta.exists() and not ruta.is_symlink():
            try:
                propio = json_estricto(ruta.read_text(encoding="utf-8"))
                if propio.get("token") == token:
                    ruta.unlink()
            except (OSError, ValueError, AttributeError):
                LOG.error("No se pudo retirar el bloqueo; revisar manualmente")


## 7. Ejecución y evidencia

La simulación no crea salidas, informes ni bloqueo. La ejecución real valida configuración, adquiere lock, recorre documentos y escribe un informe con código. Cada informe tiene identidad nueva.

Los documentos tienen contenido repetible. Los informes y sus horas son evidencia de cada ejecución y deliberadamente cambian.

In [ ]:
def ejecutar_lote(ruta_config, simular=True):
    """No destruye entradas. Simulación sin salidas, informes ni bloqueo."""
    if type(simular) is not bool:
        raise TypeError("simular debe ser booleano")
    inicio = datetime.now(timezone.utc).isoformat()
    reloj = time.perf_counter()
    cfg = cargar_config(ruta_config)
    plantilla = cargar_plantilla(cfg["plantilla"])
    def recorrer():
        vistos = set()
        seleccion = seleccionar(cfg)
        elementos = [evaluar_elemento(p, cfg, plantilla, vistos, simular) for p in seleccion]
        conteo = dict(Counter(f["estado"] for f in elementos))
        codigo = 10 if any(conteo.get(k, 0) for k in ("invalido", "duplicado", "conflicto")) else 0
        return {"version": VERSION, "simulacion": simular,
                "inicio_utc": inicio,
                "fin_utc": datetime.now(timezone.utc).isoformat(),
                "duracion_s": time.perf_counter() - reloj,
                "seleccionados": len(seleccion), "conteo": conteo,
                "elementos": elementos, "codigo": codigo, "informe": None}
    LOG.info("Inicio version=%s python=%s cwd=%s pid=%s", VERSION, sys.executable, Path.cwd(), os.getpid())
    if simular:
        return recorrer()
    with bloqueo_local(cfg["estado"]):
        resultado = recorrer()
        nombre = "ejecucion_" + uuid.uuid4().hex + ".json"
        ruta_informe = cfg["informes"] / nombre
        resultado["informe"] = str(ruta_informe)
        with ruta_informe.open("x", encoding="utf-8", newline="\n") as f:
            json.dump(resultado, f, ensure_ascii=False, indent=2)
        resultado["informe"] = str(ruta_informe)
        return resultado


In [ ]:
simulacion_demo = ejecutar_lote(config_demo, simular=True)
print(simulacion_demo["conteo"], "código", simulacion_demo["codigo"])
assert simulacion_demo["conteo"] == {"previsto":3,"invalido":2}
assert not list(cfg_demo["salida"].iterdir())
assert not list(cfg_demo["informes"].iterdir())
assert not (cfg_demo["estado"] / "ejecucion.lock").exists()


In [ ]:
primera_demo = ejecutar_lote(config_demo, simular=False)
print(primera_demo["conteo"], "código", primera_demo["codigo"])
print("Informe:", primera_demo["informe"])
assert primera_demo["conteo"] == {"creado":3,"invalido":2}
assert primera_demo["codigo"] == 10
assert json.loads(Path(primera_demo["informe"]).read_text(encoding="utf-8")) == primera_demo


In [ ]:
documentos_antes = {p.name:p.read_bytes() for p in cfg_demo["salida"].iterdir()}
segunda_demo = ejecutar_lote(config_demo, simular=False)
print(segunda_demo["conteo"])
assert segunda_demo["conteo"] == {"omitido":3,"invalido":2}
assert documentos_antes == {p.name:p.read_bytes() for p in cfg_demo["salida"].iterdir()}
assert len(list(cfg_demo["informes"].glob("*.json"))) == 2
assert entrada_antes == {p.name:p.read_bytes() for p in entrada_demo.iterdir() if p.is_file()}


### Resultado y límites

![Estados de cinco documentos en las ejecuciones](recursos/estados.png)

El código 10 es esperado: hay dos entradas inválidas. No es un fallo de instalación de Python. Para un lote sin incidencias deben corregirse esas entradas y contrastarse el contrato.

Un código 0 de lote vacío no demuestra que una fuente previa haya entregado todos los datos.

## 8. La CLI real como contrato de programación

La aplicación configura stdout y stderr como UTF-8 y utiliza códigos documentados. La cuenta programada debe poder leer el script y la configuración y escribir en las carpetas del proyecto.

Usamos sys.executable y una lista de argumentos en subprocess.run, sin shell=True. Las rutas con espacios siguen siendo argumentos completos.

In [ ]:
def main(argv=None):
    for canal in (sys.stdout, sys.stderr):
        if hasattr(canal, "reconfigure"):
            canal.reconfigure(encoding="utf-8")
    parser = argparse.ArgumentParser(description=__doc__)
    modo = parser.add_mutually_exclusive_group(required=True)
    modo.add_argument("--config", type=Path)
    modo.add_argument("--crear-demo", type=Path)
    parser.add_argument("--simular", action="store_true")
    args = parser.parse_args(argv)
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
    try:
        if args.crear_demo is not None:
            if args.simular:
                raise ConfigError("--simular se utiliza con --config")
            print(json.dumps({"config": str(crear_demo(args.crear_demo))}, ensure_ascii=False))
            return 0
        resultado = ejecutar_lote(args.config, simular=args.simular)
        print(json.dumps(resultado, ensure_ascii=False))
        return resultado["codigo"]
    except BloqueoOcupado as error:
        LOG.error("Bloqueo ocupado: %s", error)
        return 40
    except ConfigError as error:
        LOG.error("Configuración o dependencia de ruta: %s", error)
        return 20
    except OSError:
        LOG.exception("Fallo operativo: revisar posible resultado parcial")
        return 30
    except Exception:
        LOG.exception("Fallo inesperado: no se presenta el lote como correcto")
        return 30


In [ ]:
otro_cwd = base / "otra carpeta de trabajo"
otro_cwd.mkdir()
invocacion_demo = [sys.executable,"-u",str(MOTOR),"--config",str(config_demo),"--simular"]
cli_demo = subprocess.run(invocacion_demo, cwd=otro_cwd, capture_output=True,
                          text=True, encoding="utf-8", timeout=30)
print("Código:", cli_demo.returncode)
print("stderr tiene contexto:", "python=" in cli_demo.stderr)
res_cli_demo = json.loads(cli_demo.stdout)
assert cli_demo.returncode == 10
assert res_cli_demo["conteo"] == {"omitido":3,"invalido":2}


## 9. Tres capas de resultado

1. El programador acepta una definición.
2. La acción inicia el proceso correcto.
3. El lote termina y comunica su resultado.

Aceptar una definición no demuestra acceso a los datos ni éxito de la aplicación. La prueba completa compara intérprete, directorio, cuenta, registros, documentos y código de una ejecución concreta.

El notebook no carga configuraciones nativas.

## 10. Archivos de programación entregados

| Sistema | Archivos |
|---|---|
| Windows | ejecutar.ps1, ejecutar.bat, registrar_tarea.ps1, consultar_tarea.ps1. |
| Linux | ejecutar.sh, crontab.ejemplo, servicio y temporizador systemd. |
| macOS | Plist de ejemplo y generador que no lo instala. |

La teoría detalla los campos de Windows y los comandos de alta, inspección y retirada. Las plantillas deben adaptarse al equipo. Registro nativo de Windows y activación launchd no están probados en este entorno Linux.

In [ ]:
archivos_programacion = sorted(p.relative_to(PROYECTO).as_posix()
    for p in (PROYECTO / "programacion").rglob("*") if p.is_file())
print("\n".join(archivos_programacion))


## 11. Examinar formatos sin activar tareas

Una estructura válida no es un despliegue validado, pero permite comprobar errores antes. Leemos el plist con plistlib y las unidades con configparser. Cron se interpreta como cinco campos más un comando; las capacidades del daemon deben comprobarse en el equipo real.

In [ ]:
plist_ejemplo = plistlib.loads((PROYECTO / "programacion/macos/com.cursoauto.bloque04.plist.ejemplo").read_bytes())
assert isinstance(plist_ejemplo["ProgramArguments"], list)
assert plist_ejemplo["RunAtLoad"] is False
unidad_timer = configparser.ConfigParser(interpolation=None)
unidad_timer.read(PROYECTO / "programacion/linux/cursoauto-bloque04.timer", encoding="utf-8")
print("Calendario:", unidad_timer["Timer"]["OnCalendar"])
print("launchd:", plist_ejemplo["StartCalendarInterval"])


## 12. Qué debería demostrar un ensayo de Windows

El campo Programa identifica PowerShell o Python por ruta. Argumentos contiene parámetros y comillas de rutas cuando correspondan. Iniciar en señala una carpeta. La cuenta y condiciones están verificadas.

El wrapper debe esperar al hijo y devolver su ExitCode. La tarea de clase solo se registra con -Registrar y no reemplaza una existente. No cambia políticas de ejecución ni guarda contraseñas.

Para tareas corporativas, el ensayo requiere seguir las condiciones de administración del equipo.

## Ejercicios

Las tareas no registran ningún calendario. Las que procesan datos usan demos independientes dentro de nuestro contenedor temporal. Completa cada respuesta y activa su bandera. El significado de un código o el contenido de una explicación requiere además revisión humana.

## Ejercicio 01 · Selección con frontera

Define `nombres_ej1` con los nombres que seleccionar(cfg_demo) incluye. Guarda `ignorados_ej1` con nombres de archivos directos que no se seleccionan.

**Pista:** No confundas un TXT ignorado con un JSON inválido.

In [ ]:
nombres_ej1 = []  # TODO
ignorados_ej1 = []

In [ ]:
completado_ej01 = False
if completado_ej01:
    assert len(nombres_ej1)==5
    assert ignorados_ej1==["LEEME.txt"]
    print("Ejercicio 01: controles superados")
else:
    print("Ejercicio 01: pendiente; controles omitidos")

**Justifica:** explica qué evidencia aporta tu resultado y qué falta para un despliegue nativo.

## Ejercicio 02 · Documento válido y dato no admitido

Valida dato_inicial en `valido_ej2`. Guarda `rechazo_ej2=True` si la función rechaza una copia con importe "NaN".

**Pista:** El contrato requiere un importe decimal textual positivo.

In [ ]:
valido_ej2 = None  # TODO
rechazo_ej2 = False

In [ ]:
completado_ej02 = False
if completado_ej02:
    assert valido_ej2["importe"]=="1200000.00"
    assert rechazo_ej2 is True
    assert dato_inicial["importe"]=="1200000.00"
    print("Ejercicio 02: controles superados")
else:
    print("Ejercicio 02: pendiente; controles omitidos")

**Justifica:** explica qué evidencia aporta tu resultado y qué falta para un despliegue nativo.

## Ejercicio 03 · Contenido repetible de plantilla

Genera `texto_a_ej3` y `texto_b_ej3` con la misma plantilla y documento validado. Explica en `razon_ej3` por qué la hora de ejecución pertenece al informe operativo.

**Pista:** La hora dentro del documento impediría la igualdad al repetir.

In [ ]:
texto_a_ej3 = None  # TODO
texto_b_ej3 = None
razon_ej3 = ""

In [ ]:
completado_ej03 = False
if completado_ej03:
    assert texto_a_ej3==texto_b_ej3
    assert VERSION in texto_a_ej3
    assert len(razon_ej3)>40
    print("Ejercicio 03: controles superados")
else:
    print("Ejercicio 03: pendiente; controles omitidos")

**Justifica:** explica qué evidencia aporta tu resultado y qué falta para un despliegue nativo.

## Ejercicio 04 · Simulación sin escrituras

Crea la demo propia indicada y define `sim_ej4` con ejecutar_lote en simulación. Comprueba que no hay documentos, informes ni lock.

**Pista:** Las carpetas ya se crean con la demo, pero el ensayo no debe añadir archivos.

In [ ]:
config_ej4 = crear_demo(base / "ej04")
sim_ej4 = None  # TODO

In [ ]:
completado_ej04 = False
if completado_ej04:
    assert sim_ej4["conteo"]=={"previsto":3,"invalido":2}
    assert not list((config_ej4.parent/"salida").iterdir())
    assert not list((config_ej4.parent/"informes").iterdir())
    assert not (config_ej4.parent/"estado/ejecucion.lock").exists()
    print("Ejercicio 04: controles superados")
else:
    print("Ejercicio 04: pendiente; controles omitidos")

**Justifica:** explica qué evidencia aporta tu resultado y qué falta para un despliegue nativo.

## Ejercicio 05 · Lote real con incidencias

Procesa su demo propia y guarda `real_ej5`. Define `suma_ej5` como suma de los conteos y `informe_ej5` como JSON leído desde el informe.

**Pista:** El código 10 es parte del resultado esperado, no un error de tu solución.

In [ ]:
config_ej5 = crear_demo(base / "ej05")
real_ej5 = None  # TODO
suma_ej5 = None
informe_ej5 = None

In [ ]:
completado_ej05 = False
if completado_ej05:
    assert real_ej5["codigo"]==10
    assert real_ej5["conteo"]=={"creado":3,"invalido":2}
    assert suma_ej5==real_ej5["seleccionados"]==5
    assert informe_ej5==real_ej5
    print("Ejercicio 05: controles superados")
else:
    print("Ejercicio 05: pendiente; controles omitidos")

**Justifica:** explica qué evidencia aporta tu resultado y qué falta para un despliegue nativo.

## Ejercicio 06 · Repetición y evidencia nueva

En una demo propia ejecuta dos veces. Guarda la segunda como `segunda_ej6` y `n_informes_ej6`. Comprueba que hay tres documentos y dos informes.

**Pista:** No todo el sistema tiene que quedar idéntico: los informes representan cada ejecución.

In [ ]:
config_ej6 = crear_demo(base / "ej06")
# TODO: dos ejecuciones reales
segunda_ej6 = None
n_informes_ej6 = None

In [ ]:
completado_ej06 = False
if completado_ej06:
    assert segunda_ej6["conteo"]=={"omitido":3,"invalido":2}
    assert n_informes_ej6==2
    assert len(list((config_ej6.parent/"salida").glob("*.md")))==3
    print("Ejercicio 06: controles superados")
else:
    print("Ejercicio 06: pendiente; controles omitidos")

**Justifica:** explica qué evidencia aporta tu resultado y qué falta para un despliegue nativo.

## Ejercicio 07 · Conflicto sin sobrescritura

Se prepara una demo y se altera deliberadamente una salida de ese entorno ficticio. Repite el lote como `conflicto_ej7` y comprueba que ese contenido no es reemplazado.

**Pista:** La modificación del fixture solo simula un cambio externo en nuestros datos de aula.

In [ ]:
config_ej7 = crear_demo(base / "ej07")
ejecutar_lote(config_ej7,simular=False)
salida_ej7 = config_ej7.parent / "salida/doc_DEMO-001_2025.md"
salida_ej7.write_text("Cambio externo de aula",encoding="utf-8")
conflicto_ej7 = None  # TODO

In [ ]:
completado_ej07 = False
if completado_ej07:
    assert conflicto_ej7["conteo"]=={"conflicto":1,"omitido":2,"invalido":2}
    assert salida_ej7.read_text(encoding="utf-8")=="Cambio externo de aula"
    print("Ejercicio 07: controles superados")
else:
    print("Ejercicio 07: pendiente; controles omitidos")

**Justifica:** explica qué evidencia aporta tu resultado y qué falta para un despliegue nativo.

## Ejercicio 08 · No ejecutar dos escritores

Con su demo propia, mantén un bloqueo usando el contexto y captura BloqueoOcupado al llamar al lote real. Guarda `ocupado_ej8=True` y comprueba retirada al salir del contexto propio.

**Pista:** No borres un lock de otra ejecución para hacer pasar la prueba.

In [ ]:
config_ej8 = crear_demo(base / "ej08")
ocupado_ej8 = False
# TODO: contexto de bloqueo e intento de ejecución

In [ ]:
completado_ej08 = False
if completado_ej08:
    assert ocupado_ej8 is True
    assert not (config_ej8.parent/"estado/ejecucion.lock").exists()
    assert not list((config_ej8.parent/"salida").iterdir())
    print("Ejercicio 08: controles superados")
else:
    print("Ejercicio 08: pendiente; controles omitidos")

**Justifica:** explica qué evidencia aporta tu resultado y qué falta para un despliegue nativo.

## Ejercicio 09 · CLI desde otra carpeta

Invoca el MOTOR mediante sys.executable, -u, --config absoluto y --simular desde una carpeta diferente. Guarda `proceso_ej9` y `resultado_ej9`. Usa lista de argumentos, sin shell=True.

**Pista:** subprocess.run debe capturar salidas como UTF-8 y tener un tiempo límite.

In [ ]:
config_ej9 = crear_demo(base / "ej09 con espacios")
cwd_ej9 = base / "cwd_ej09"
cwd_ej9.mkdir()
proceso_ej9 = None  # TODO
resultado_ej9 = None

In [ ]:
completado_ej09 = False
if completado_ej09:
    assert proceso_ej9.returncode==10
    assert resultado_ej9["conteo"]=={"previsto":3,"invalido":2}
    assert "python=" in proceso_ej9.stderr
    assert not list((config_ej9.parent/"salida").iterdir())
    print("Ejercicio 09: controles superados")
else:
    print("Ejercicio 09: pendiente; controles omitidos")

**Justifica:** explica qué evidencia aporta tu resultado y qué falta para un despliegue nativo.

## Ejercicio 10 · Separar los campos de Windows

Define `accion_ej10` con programa, argumentos e iniciar_en, usando el ejemplo de C:/CursoAuto/bloque04. El programa será el PowerShell de ruta completa y el archivo del wrapper irá en Argumentos. No registres una tarea.

**Pista:** La ruta del script no ocupa el campo Programa cuando se lanza PowerShell.

In [ ]:
accion_ej10 = {"programa":"", "argumentos":"", "iniciar_en":""}

In [ ]:
completado_ej10 = False
if completado_ej10:
    assert PureWindowsPath(accion_ej10["programa"]).is_absolute()
    assert accion_ej10["programa"].endswith("powershell.exe")
    assert all(x in accion_ej10["argumentos"] for x in ["-NoProfile","-NonInteractive","-File","ejecutar.ps1"])
    assert accion_ej10["iniciar_en"]=="C:/CursoAuto/bloque04"
    print("Ejercicio 10: controles superados")
else:
    print("Ejercicio 10: pendiente; controles omitidos")

**Justifica:** explica qué evidencia aporta tu resultado y qué falta para un despliegue nativo.

## Ejercicio 11 · Interpretar cron y timer

Define `cron_ej11` para cada día a las 09:00 como cinco campos. Lee el timer de ejemplo en `timer_ej11` con configparser y explica la recuperación de Persistent en `limite_ej11`.

**Pista:** Persistent no reproduce una ejecución por cada instante perdido.

In [ ]:
cron_ej11 = ""
timer_ej11 = configparser.ConfigParser(interpolation=None)
# TODO: leer el archivo del timer
limite_ej11 = ""

In [ ]:
completado_ej11 = False
if completado_ej11:
    assert cron_ej11.split()==["0","9","*","*","*"]
    assert timer_ej11["Timer"]["OnCalendar"]=="*-*-* 09:00:00"
    assert timer_ej11.getboolean("Timer","Persistent") is True
    assert len(limite_ej11)>45
    print("Ejercicio 11: controles superados")
else:
    print("Ejercicio 11: pendiente; controles omitidos")

**Justifica:** explica qué evidencia aporta tu resultado y qué falta para un despliegue nativo.

## Ejercicio 12 · Construir un plist como datos

Define `configuracion_ej12` con Label propio, ProgramArguments como lista, WorkingDirectory, StartCalendarInterval con Hour 9 y Minute 0 y RunAtLoad False. Usa rutas absolutas de este laboratorio para validar estructura, sin activarlo. Serializa en `bytes_plist_ej12`.

**Pista:** Cada argumento ocupa un elemento. No añadas comillas de shell dentro de una ruta.

In [ ]:
configuracion_ej12 = {}  # TODO
bytes_plist_ej12 = None

In [ ]:
completado_ej12 = False
if completado_ej12:
    assert plistlib.loads(bytes_plist_ej12)==configuracion_ej12
    assert len(configuracion_ej12["ProgramArguments"])==5
    assert configuracion_ej12["StartCalendarInterval"]=={"Hour":9,"Minute":0}
    assert configuracion_ej12["RunAtLoad"] is False
    print("Ejercicio 12: controles superados")
else:
    print("Ejercicio 12: pendiente; controles omitidos")

**Justifica:** explica qué evidencia aporta tu resultado y qué falta para un despliegue nativo.

## Cierre y retirada de los datos temporales

La práctica no ha instalado ninguna tarea. Los proyectos creados viven solo dentro de su contenedor temporal. Puedes retirarlo cuando termines de inspeccionar; guarda tus respuestas en el notebook. No uses estas rutas temporales como configuración programada persistente.

In [ ]:
retirar_temporal = False
if retirar_temporal:
    temporal.cleanup()
    print("Datos temporales propios retirados")
else:
    print("Disponibles durante esta sesión:", base)


## Evaluación y fuentes

Dos puntos por dimensión: contrato de lote, interpretación de estados/códigos, evidencia de repetición, contexto de ejecución y configuración sin efectos inesperados. Las explicaciones requieren revisión humana.

Fuentes primarias en las diapositivas y en GUIA_PROGRAMACION.md: Microsoft ScheduledTasks y schtasks, Apple launchd (documentación archivada y manuales del equipo), crontab(5), systemd.timer y Python 3.11. Los códigos y políticas de aula son propios del ejemplo. Revisión: 03/11/2026.